In [ ]:
import os, sys, pathlib
RAIZ = pathlib.Path.cwd()
while not (RAIZ / 'scripts').is_dir() and RAIZ != RAIZ.parent:
    RAIZ = RAIZ.parent
os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ / 'scripts'))
print('raiz do projeto:', RAIZ)

# nb04 — VPI interativa (níveis de pressão e isentrópico)

**O que este notebook PRODUZ (tudo sobre dados JÁ baixados — abre na hora, sem download):**

1. **Mapa interativo** de VPI / θ / T / vento, em qualquer **nível padrão** (925→300 hPa),
   varrendo **todo o período** com um slider — **sempre com fronteiras/costa e cidades**.
2. **Sobreposições**: precipitação (contornos) e vetores de vento sobre o campo escolhido.
3. **Animação** (player play/pause/scrub) do campo num nível, no período todo.
4. **Superfície isentrópica**: escolhe uma θ (ex. 300, 310, 320 K) e vê pressão/VPI nela.
5. **Qualidade da interpolação**: grade nativa × suavizada, espaçamento da grade, e um
   **corte vertical** (lat × pressão) para ver a estrutura e o quão grosseiros são os 7 níveis.

> Fontes: `_era5_pl_brasil_rs.nc` (PV, T), `_era5_uvwt_abrmai.nc` (vento), `_era5_meteo_*.nc` (precip).

## 1) Carregar e MOSTRAR o que tem (período, níveis, grade)

In [ ]:
import os, numpy as np, pandas as pd, xarray as xr, matplotlib.pyplot as plt
import matplotlib.animation as manim
import cartopy.crs as ccrs
import ipywidgets as W; from IPython.display import display, HTML
import importlib, ondas_config, viz_helpers as vh
importlib.reload(ondas_config); importlib.reload(vh)
%matplotlib inline
RES='resultados'; DADOS='dados'
for _v in ('ds_pl','ds_uv','ds_pr'):
    try: globals()[_v].close()
    except Exception: pass
ds_pl = vh.abre_nc(f'{DADOS}/era5/_era5_pl_brasil_rs.nc')
LAT,LON,TDIM,LEV = vh.coords_nc(ds_pl)
uvp=f'{DADOS}/era5/_era5_uvwt_abrmai.nc'; ds_uv = vh.abre_nc(uvp) if os.path.exists(uvp) else None
prp=f'{DADOS}/era5/_era5_meteo_2024-04-27_2024-05-15.nc'; ds_pr = vh.abre_nc(prp) if os.path.exists(prp) else None
TIMES=pd.to_datetime(ds_pl[TDIM].values)
NIVEIS=[int(x) for x in ds_pl[LEV].values]
dlat=abs(float(ds_pl[LAT][1]-ds_pl[LAT][0])); dlon=abs(float(ds_pl[LON][1]-ds_pl[LON][0]))
print('PERÍODO :', TIMES.min(), '->', TIMES.max(), f'({len(TIMES)} tempos, passo {int((TIMES[1]-TIMES[0]).seconds/3600)} h)')
print('NÍVEIS  :', NIVEIS, 'hPa')
print('GRADE   : lat', float(ds_pl[LAT].min()),'..',float(ds_pl[LAT].max()), f'(Δ={dlat}°) | lon', float(ds_pl[LON].min()),'..',float(ds_pl[LON].max()), f'(Δ={dlon}°)')
print('VENTO   :', 'sim' if ds_uv is not None else 'NÃO', '| PRECIP:', 'sim' if ds_pr is not None else 'NÃO')

## 2) Funções de campo (VISÍVEIS — nada escondido)
VPI em PVU (negativa = ciclônica no HS). θ potencial. Vento e precip por tempo mais próximo.

In [ ]:
KAPPA=0.2854
def _sel_t(ds, when):
    _,_,td,_=vh.coords_nc(ds); return ds.sel({td:np.datetime64(when)}, method='nearest')
def campo_vpi(when, plev):
    s=_sel_t(ds_pl, when).sel({LEV:plev}); return (s['pv'].values*1e6)       # PVU
def campo_theta(when, plev):
    s=_sel_t(ds_pl, when).sel({LEV:plev}); return (s['t'].values*(1000.0/plev)**KAPPA)
def campo_T(when, plev):
    return _sel_t(ds_pl, when).sel({LEV:plev})['t'].values-273.15
def vento(when, plev):
    if ds_uv is None: return None
    la,lo,td,lv=vh.coords_nc(ds_uv); s=_sel_t(ds_uv, when).sel({lv:plev})
    return ds_uv[la].values, ds_uv[lo].values, s['u'].values, s['v'].values
def precip(when):
    if ds_pr is None: return None
    la,lo,td,lv=vh.coords_nc(ds_pr); s=_sel_t(ds_pr, when)
    v=next((n for n in ('mtpr','avg_tprate') if n in ds_pr), None)
    if v is None: return None
    return ds_pr[la].values, ds_pr[lo].values, s[v].values*3600.0   # mm/h
CAMPOS={'VPI (PVU)':(campo_vpi,'RdBu',-2,2),'θ (K)':(campo_theta,'turbo',280,360),'T (°C)':(campo_T,'turbo',-60,25)}
VISTA={'sul':dict(latmin=-40,latmax=-18,lonmin=-68,lonmax=-42),
       'rs':dict(latmin=-36,latmax=-25,lonmin=-60,lonmax=-47),
       'brasil':dict(latmin=-34,latmax=6,lonmin=-74,lonmax=-34)}

## 3) Mapa interativo — variável × nível × tempo, com fronteiras e sobreposições

In [ ]:
def mapa(nome_var, plev, ti, vista, sobrepor_precip, vetores_vento):
    when=TIMES[ti]; f,cmap,vmin,vmax=CAMPOS[nome_var]
    campo=f(when, plev); cx=VISTA[vista]
    fig,ax=vh.novo_mapa(cx, figsize=(8.5,8))
    m=ax.pcolormesh(ds_pl[LON].values, ds_pl[LAT].values, campo, cmap=cmap,
                    vmin=vmin, vmax=vmax, shading='auto', transform=ccrs.PlateCarree(), zorder=1)
    fig.colorbar(m,ax=ax,fraction=0.04,label=nome_var)
    if sobrepor_precip and ds_pr is not None:
        pla,plo,pp=precip(when)
        cs=ax.contour(plo,pla,pp,levels=[0.5,1,2,4,8],colors='k',linewidths=0.8,alpha=0.7,
                      transform=ccrs.PlateCarree(), zorder=3)
        ax.clabel(cs,fontsize=7,fmt='%g')
    if vetores_vento and ds_uv is not None:
        ula,ulo,u,v=vento(when, plev); st=max(1,len(ulo)//30)
        ax.quiver(ulo[::st],ula[::st],u[::st,::st],v[::st,::st],scale=500,width=0.002,color='#333',
                  transform=ccrs.PlateCarree(), zorder=4)
    vh.extensao(ax, cx)
    ax.set_title(f"{nome_var} em {plev} hPa - {when:%Y-%m-%d %HZ}"); plt.show()
W.interact(mapa,
  nome_var=W.Dropdown(options=list(CAMPOS),value='VPI (PVU)',description='variável'),
  plev=W.Dropdown(options=NIVEIS,value=925 if 925 in NIVEIS else NIVEIS[0],description='nível hPa'),
  ti=W.IntSlider(min=0,max=len(TIMES)-1,value=0,description='tempo'),
  vista=W.Dropdown(options=list(VISTA),value='sul',description='vista'),
  sobrepor_precip=W.Checkbox(value=True,description='precip (contornos)'),
  vetores_vento=W.Checkbox(value=False,description='vetores de vento'));

## 4) Animação do período (um nível, com player)
Escolha variável+nível abaixo e rode; o player deixa dar play/pause/scrub.

In [ ]:
VAR_ANIM='VPI (PVU)'; NIVEL_ANIM=925; VISTA_ANIM='sul'; PASSO_ANIM=2
f,cmap,vmin,vmax=CAMPOS[VAR_ANIM]; cx=VISTA[VISTA_ANIM]
idx=list(range(0,len(TIMES),PASSO_ANIM))
fig,ax=vh.novo_mapa(cx, figsize=(8,7.5))
m=ax.pcolormesh(ds_pl[LON].values,ds_pl[LAT].values,f(TIMES[idx[0]],NIVEL_ANIM),
                cmap=cmap,vmin=vmin,vmax=vmax,shading='auto',transform=ccrs.PlateCarree(),zorder=1)
fig.colorbar(m,ax=ax,fraction=0.04,label=VAR_ANIM)
vh.extensao(ax,cx); tit=ax.set_title('')
def upd(k):
    when=TIMES[idx[k]]; m.set_array(f(when,NIVEL_ANIM).ravel())
    tit.set_text(f'{VAR_ANIM} {NIVEL_ANIM} hPa - {when:%Y-%m-%d %HZ}'); return m,tit
ani=manim.FuncAnimation(fig,upd,frames=len(idx),interval=250,blit=False); plt.close(fig)
print(f'{len(idx)} quadros de {len(TIMES)} tempos (passo {PASSO_ANIM})')
HTML(ani.to_jshtml())

## 5) Superfície ISENTRÓPICA — pressão e VPI numa θ escolhida
Interpola a coluna para a superfície θ0 (adiabática). Mostra em que pressão ela está e a VPI nela.

In [ ]:
def _iso(campo_lev, theta_lev, th0):
    # campo_lev, theta_lev: (nlev,nlat,nlon). Retorna campo em theta=th0 (interp linear por coluna).
    order=np.argsort(theta_lev,axis=0)
    th=np.take_along_axis(theta_lev,order,axis=0); fl=np.take_along_axis(campo_lev,order,axis=0)
    idx=np.sum(th<th0,axis=0); nb=th.shape[0]
    i1=np.clip(idx,1,nb-1)[None]; i0=i1-1
    th0v=np.take_along_axis(th,i0,0)[0]; th1v=np.take_along_axis(th,i1,0)[0]
    f0=np.take_along_axis(fl,i0,0)[0]; f1=np.take_along_axis(fl,i1,0)[0]
    w=(th0-th0v)/np.where(th1v==th0v,np.nan,(th1v-th0v)); res=f0+w*(f1-f0)
    res[(idx==0)|(idx==nb)]=np.nan; return res
def iso_mapa(th0, ti, vista, qual):
    when=TIMES[ti]; s=_sel_t(ds_pl, when); cx=VISTA[vista]
    P=np.array(NIVEIS,dtype=float)[:,None,None]
    T=s['t'].transpose(LEV,LAT,LON).values; theta=T*(1000.0/P)**KAPPA
    pv=s['pv'].transpose(LEV,LAT,LON).values*1e6
    plev3=np.broadcast_to(P,theta.shape)
    campo = _iso(pv,theta,th0) if qual=='VPI (PVU)' else _iso(plev3,theta,th0)
    cmap,vmin,vmax=(('RdBu',-2,2) if qual=='VPI (PVU)' else ('viridis_r',200,1000))
    fig,ax=vh.novo_mapa(cx, figsize=(8.5,8))
    m=ax.pcolormesh(ds_pl[LON].values,ds_pl[LAT].values,campo,cmap=cmap,vmin=vmin,vmax=vmax,
                    shading='auto',transform=ccrs.PlateCarree(),zorder=1)
    fig.colorbar(m,ax=ax,fraction=0.04,label=qual+(' na θ' if qual=='VPI (PVU)' else ' (hPa) da θ'))
    vh.extensao(ax,cx)
    ax.set_title(f'Superfície θ={th0} K - {qual} - {when:%Y-%m-%d %HZ}'); plt.show()
W.interact(iso_mapa,
  th0=W.IntSlider(min=295,max=345,step=5,value=315,description='θ (K)'),
  ti=W.IntSlider(min=0,max=len(TIMES)-1,value=0,description='tempo'),
  vista=W.Dropdown(options=list(VISTA),value='sul',description='vista'),
  qual=W.Dropdown(options=['VPI (PVU)','pressão (hPa)'],value='VPI (PVU)',description='mostrar'));

## 6) Qualidade da interpolação
Grade **nativa** (pcolormesh, sem suavizar) × **suavizada** (contourf), o espaçamento real,
e um **corte vertical** (lat × pressão) para ver a estrutura e o quão grosseiros são os 7 níveis.

In [ ]:
def qualidade(plev, ti, lon_corte):
    when=TIMES[ti]; campo=campo_vpi(when,plev); cx=VISTA['sul']
    la=ds_pl[LAT].values; lo=ds_pl[LON].values
    fig=plt.figure(figsize=(17,5))
    _,ax0=vh.novo_mapa(cx, fig=fig, subplot=(1,3,1))
    _,ax1=vh.novo_mapa(cx, fig=fig, subplot=(1,3,2))
    ax2=fig.add_subplot(1,3,3)
    ax0.pcolormesh(lo,la,campo,cmap='RdBu',vmin=-2,vmax=2,shading='nearest',
                   transform=ccrs.PlateCarree(),zorder=1)
    ax0.set_title(f'NATIVA (sem interp) - {plev} hPa')
    ax1.contourf(lo,la,campo,levels=np.linspace(-2,2,21),cmap='RdBu',extend='both',
                 transform=ccrs.PlateCarree(),zorder=1)
    ax1.set_title('SUAVIZADA (contourf)')
    ax0.plot([lon_corte,lon_corte],[cx['latmin'],cx['latmax']],'k--',lw=1,
             transform=ccrs.PlateCarree(),zorder=7)
    vh.extensao(ax0,cx); vh.extensao(ax1,cx)
    s=_sel_t(ds_pl,when); iL=int(np.argmin(np.abs(lo-lon_corte)))
    P=np.array(NIVEIS,dtype=float); pv2=s['pv'].transpose(LEV,LAT,LON).values[:,:,iL]*1e6
    cf=ax2.contourf(la,P,pv2,levels=np.linspace(-2,2,21),cmap='RdBu',extend='both')
    ax2.scatter(np.repeat(la,len(P)),np.tile(P,len(la)),s=0.5,c='k',alpha=0.15)
    ax2.invert_yaxis(); ax2.set_xlabel('lat'); ax2.set_ylabel('pressão (hPa)')
    ax2.set_title(f'CORTE vertical em lon={lon_corte}° (pontos = níveis reais)')
    fig.colorbar(cf,ax=ax2,fraction=0.04,label='VPI (PVU)')
    print(f'grade: Δlat={abs(la[1]-la[0]):.3f}°  Δlon={abs(lo[1]-lo[0]):.3f}°  |  {len(NIVEIS)} níveis: {NIVEIS}')
    plt.show()
W.interact(qualidade,
  plev=W.Dropdown(options=NIVEIS,value=700 if 700 in NIVEIS else NIVEIS[0],description='nível hPa'),
  ti=W.IntSlider(min=0,max=len(TIMES)-1,value=0,description='tempo'),
  lon_corte=W.FloatSlider(min=-68,max=-42,step=1,value=-52,description='lon do corte'));